# Eigenvalues & Decomposition

**Module**: 00 Mathematical Foundations · **Topic**: Linear Algebra · **Notebook**: 02  
**Prerequisites**: None · **Estimated time**: 60 min  
**Priority band**: P1 Diagnostics — *Does this person have the condition? How sure are we?*  
**Goal**: `predict` — Predict — the model's job is accuracy on unseen patients; coefficients are not mechanisms.  
**GDS division**: 3 Computing with data  
**Dataset**: `screening`

## Learning Objectives

By the end of this notebook you will be able to:

1. Say what an eigenvector and an eigenvalue of a covariance matrix are: a direction in which the cloud of patients is only stretched, and how much it is stretched.
2. Find the top eigenpairs from scratch (power iteration with deflation), check them against `np.linalg.eigh`, and rebuild the matrix from them as $S = V\Lambda V^\top$.
3. Read the eigen-decomposition of a screening panel: what the loadings and the near-zero eigenvalue reveal about a duplicated test.
4. Show that rotating a panel rotates its eigenvectors and leaves its eigenvalues alone, and that PCA, `eigh` of the covariance and the SVD of the data are the same computation.
5. Recognise when the decomposition misleads: unstandardised units, crowded eigenvalues with too few patients, and the fact that big variance is not the same as disease signal.

In [ ]:
import sys
from pathlib import Path
# Jupyter starts the kernel in this notebook's folder; put the repo root on the path so shared/ and datasets/ import
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from shared.plotting import set_style, PALETTE, grid
from shared.explanation import rfi, clinical_reading, three_registers
from datasets import load_screening
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

set_style()
SEED = 42
np.random.seed(SEED)
HEALTHY, DISEASED = PALETTE[0], PALETTE[1]   # one colour per disease status, used in every figure

---
## 1. The Clinical Problem

🗣️ **Plain** — A GP's screening panel has twelve test results per patient. The band's question is *does this person have the condition, and how sure are we?* Before anyone fits a model, a sensible first step is to ask: **in which directions do patients actually differ from one another?** Twelve tests do not give twelve independent pieces of information. Two tests may rise and fall together, and one may simply be a second copy of another. The directions along which the cloud of patients is widest are where the information is; the directions where it is almost flat are where tests repeat each other. Eigenvalues and eigenvectors are the tool that finds those directions and measures how wide the cloud is along each.

**The data.** 2,000 synthetic screening patients, twelve tests, 15% with the condition. By construction (the generator's `truth`), two tests are strongly informative (`bm_01`, `bm_02`), three are weakly informative, the rest are noise, and `bm_12` is a near-copy of `bm_01`. We keep the generator's answer to one side and check the method against it afterwards. The first split below sets aside 600 test patients; everything we learn (means, spreads, eigenvectors) comes from the other 1,400.

In [ ]:
# VISUAL 1 — the raw clinical data
df, truth = load_screening(return_truth=True)           # truth = what the generator built in, for checking later
BM = [c for c in df.columns if c.startswith("bm_")]
y = df["disease"].to_numpy()
X_raw = df[BM].to_numpy()                               # patients x tests
# Split first: every mean, SD and eigenvector below is learned on TRAIN patients only
idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=0.3, stratify=y, random_state=SEED)
mu, sd = X_raw[idx_tr].mean(0), X_raw[idx_tr].std(0)
Z = (X_raw - mu) / sd                                   # standardised panel (SD units)
Ztr, Zte, ytr, yte = Z[idx_tr], Z[idx_te], y[idx_tr], y[idx_te]
print(f"{len(df)} patients, {len(BM)} tests, prevalence {y.mean():.1%}; train {len(idx_tr)}, test {len(idx_te)}")
print("generator truth:", truth)

s, (a, b, c) = plt.subplots(1, 3, figsize=(18, 5.2))
col = np.where(y == 1, DISEASED, HEALTHY)
order = np.argsort(y)                                    # draw diseased patients last so they stay visible
a.scatter(Z[order, 0], Z[order, 11], c=col[order], s=10, alpha=0.6)
a.set_xlabel("bm_01 (SD units)"); a.set_ylabel("bm_12 (SD units)")
a.set_title("bm_12 is a copy of bm_01: the cloud is a line")
b.scatter(Z[order, 0], Z[order, 1], c=col[order], s=10, alpha=0.6)
b.set_xlabel("bm_01 (SD units)"); b.set_ylabel("bm_02 (SD units)")
b.set_title("Two strong tests: diseased patients sit up and to the right")
sns.heatmap(np.corrcoef(Ztr.T), vmin=-1, vmax=1, cmap="vlag", ax=c, xticklabels=[x[3:] for x in BM], yticklabels=[x[3:] for x in BM], cbar_kws={"label": "correlation"})
c.set_title("Almost every pair is uncorrelated, except 01 and 12")
c.set_xlabel("test number"); c.set_ylabel("test number")
plt.tight_layout(); plt.show()

---
## 2. What Is Eigenvalues & Decomposition?

🗣️ **Plain**

Picture the patients as a cloud of dots, one dot per person, floating in a room where each wall is a test. Most pushes you could give the whole cloud (squash it, shear it, stretch it) would turn it and bend it. But for any such push there are a few special directions along which the cloud is **only stretched or shrunk, never turned**. Those directions are the **eigenvectors** ("own directions"). How much the cloud is stretched along each one is its **eigenvalue**. For a cloud of patients the useful push is "how the tests vary together", and the eigenvectors are then the cloud's own axes, like the long and short axes of a rugby ball. The eigenvalue tells you how long each axis is. An axis with a large eigenvalue is a combination of tests on which patients differ a lot. An axis with an eigenvalue near zero is a combination on which every patient looks the same, which is exactly what you get when one test is a copy of another.

💻 **Developer**

```text
S  = cov(Z)                      # d x d, symmetric, from standardised TRAIN data
# from scratch (top-k): power iteration + deflation
for i in 1..k:
    v = random unit vector
    repeat n_iter times:  v = S v ; v = v / ||v||
    lambda_i = v' S v ;  v_i = v
    S = S - lambda_i v_i v_i'            # deflate
# library:
lam, V = np.linalg.eigh(S)               # symmetric solver, ascending order; flip to descending
scores = Z_new @ V                       # coordinates of NEW patients on the TRAIN eigenvectors
```

Cost: `eigh` is $O(d^3)$ for a $d\times d$ matrix (trivial at $d=12$, noticeable at $d=10^4$); forming the covariance is $O(nd^2)$. Power iteration costs $O(d^2)$ per step but converges at a rate set by the ratio of neighbouring eigenvalues, so crowded eigenvalues are slow. Memory is $O(d^2)$. Eigenvectors are only defined up to sign; never compare them without fixing the sign.

🔬 **Technical**

For a square matrix $A$, a non-zero $v$ with $Av=\lambda v$ is an eigenvector with eigenvalue $\lambda$. For the sample covariance $S=\frac{1}{n-1}Z_c^\top Z_c$ of centred data, $S$ is real symmetric and positive semi-definite, so the spectral theorem gives
$$S = V\Lambda V^\top,\qquad V^\top V = I,\qquad \Lambda=\mathrm{diag}(\lambda_1\ge\dots\ge\lambda_d\ge 0).$$
$\lambda_j$ is the variance of the data projected on $v_j$, and $\sum_j\lambda_j=\mathrm{tr}\,S$ is the total variance (equal to $d$ for standardised data). $v_1=\arg\max_{\|v\|=1}v^\top S v$ (Rayleigh quotient), and each next $v_j$ maximises it over the orthogonal complement. If $Z_c=U\Sigma W^\top$ is the SVD, then $S=W\frac{\Sigma^2}{n-1}W^\top$, so the right singular vectors are the eigenvectors and $\lambda_j=\sigma_j^2/(n-1)$: PCA is this decomposition. Assumptions: finite second moments; eigenvectors are unique (up to sign) only when eigenvalues are distinct; sample eigenvectors are stable only when the gap to the neighbouring eigenvalue is large relative to $\lambda/\sqrt n$ (Davis–Kahan). Eigen-decomposition describes variance, not the outcome: nothing here uses $y$.

---
## 3. Visual Intuition

The picture below uses a 2 x 2 matrix so that everything can be drawn. The matrix $A$ sends the unit circle to an ellipse. Its eigenvectors are the ellipse's axes: they point the same way before and after and are only lengthened (by 1 and by 3). An ordinary direction comes out turned.

In [ ]:
# VISUAL 2 — concept diagram: a matrix bends the plane, but its eigenvectors only get stretched
A = np.array([[2.0, 1.0], [1.0, 2.0]])                  # a symmetric 2x2 matrix (a covariance has this shape)
lam, V = np.linalg.eigh(A)                              # eigenvalues ascending, eigenvectors in the columns
th = np.linspace(0, 2 * np.pi, 200)
circle = np.vstack([np.cos(th), np.sin(th)])            # the unit circle
ellipse = A @ circle                                    # where A sends it

fig, (a, b) = plt.subplots(1, 2, figsize=(13, 6))
for ax, title in [(a, "Before: every direction has length 1"), (b, "After: the circle becomes an ellipse whose axes are the eigenvectors")]:
    ax.set_aspect("equal"); ax.set_xlim(-3.4, 3.4); ax.set_ylim(-3.4, 3.4); ax.set_title(title, fontsize=11)
    ax.set_xlabel("test 1"); ax.set_ylabel("test 2")
a.plot(*circle, color=PALETTE[7], label="unit circle")
b.plot(*circle, color=PALETTE[7], ls=":", label="unit circle")
b.plot(*ellipse, color=PALETTE[0], label="A x unit circle")
for k, name in enumerate(["short axis", "long axis"]):
    v = V[:, k]
    a.annotate("", xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=PALETTE[1 + k], lw=2.5))
    b.annotate("", xy=lam[k] * v, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=PALETTE[1 + k], lw=2.5))
    b.plot([], [], color=PALETTE[1 + k], lw=2.5, label=f"eigenvector, stretched x{lam[k]:.0f} ({name})")
w = np.array([1.0, 0.0])                                 # an ordinary direction: it gets bent AND stretched
a.annotate("", xy=w, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=PALETTE[3], lw=2.5)); a.plot([], [], color=PALETTE[3], lw=2.5, label="ordinary direction")
b.annotate("", xy=A @ w, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=PALETTE[3], lw=2.5)); b.plot([], [], color=PALETTE[3], lw=2.5, label="ordinary direction, turned")
a.legend(loc="lower right"); b.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()
print("eigenvalues:", lam, " A v = lambda v holds:", np.allclose(A @ V, V * lam))

---
## 4. Implementation from Scratch

Power iteration: multiply a random vector by $S$ again and again, renormalising each time. Each multiplication stretches the largest eigen-direction most, so the vector swings towards it. Deflation then subtracts what was found and repeats for the next. The code runs on the 12 x 12 covariance of the standardised **training** patients, and the figure shows how fast each eigenvalue converges.

In [ ]:
# Power iteration finds the top eigenvector; deflation finds the next one. All from scratch, NumPy only.
def power_iteration(S, n_iter=200, seed=SEED):
    """Largest eigenpair of a symmetric matrix S. Cost: n_iter matrix-vector products, O(n_iter * d^2)."""
    v = np.random.default_rng(seed).normal(size=S.shape[0])   # random start direction
    v /= np.linalg.norm(v)
    history = []
    for _ in range(n_iter):
        v = S @ v                       # stretch: the biggest eigen-direction is stretched most...
        v /= np.linalg.norm(v)          # ...so after renormalising, v leans towards it more each step
        history.append(v @ S @ v)       # Rayleigh quotient: current estimate of the eigenvalue
    return v @ S @ v, v, np.array(history)

def my_eig(S, k, n_iter=500):
    """Top-k eigenpairs by power iteration + deflation (subtract each found component and repeat)."""
    S = S.copy(); vals, vecs, hist = [], [], []
    for i in range(k):
        lam, v, h = power_iteration(S, n_iter, seed=SEED + i)
        vals.append(lam); vecs.append(v); hist.append(h)
        S = S - lam * np.outer(v, v)    # deflation: remove the component just found
    return np.array(vals), np.array(vecs).T, hist

S_tr = np.cov(Ztr.T)                    # 12 x 12 covariance of the standardised TRAIN panel
vals_me, vecs_me, hist = my_eig(S_tr, 4)
w_ref, V_ref = np.linalg.eigh(S_tr); w_ref, V_ref = w_ref[::-1], V_ref[:, ::-1]   # descending order
print("mine   :", np.round(vals_me, 4))
print("eigh   :", np.round(w_ref[:4], 4))
print("|cos| between my vectors and eigh's:", np.round(np.abs((vecs_me * V_ref[:, :4]).sum(0)), 6))

fig, ax = plt.subplots(figsize=(12, 5.5))
for i, h in enumerate(hist):
    ax.plot(np.arange(1, 251), np.abs(h[:250] - w_ref[i]) + 1e-16, label=f"eigenvalue {i+1} (true {w_ref[i]:.2f})", color=PALETTE[i])
for i, h in enumerate(hist):
    print(f"eigenvalue {i+1}: steps to reach 1e-6 of eigh:", int(np.argmax(np.abs(h - w_ref[i]) < 1e-6)) + 1)
ax.set_yscale("log"); ax.set_xlabel("power-iteration step"); ax.set_ylabel("|estimate - eigh value|")
ax.set_title("The top eigenvalue takes 12 steps; the crowded ones need over 130")
ax.legend(); plt.show()

---
## 5. Library Implementation

`np.linalg.eigh` is the dense solver for symmetric matrices. It returns real eigenvalues and orthonormal eigenvectors, in ascending order. We check the two identities the rest of the notebook leans on: $V^\top V=I$ and $S=V\Lambda V^\top$. The figure asks how many eigen-directions we need to keep to rebuild the covariance.

In [ ]:
# Library version: np.linalg.eigh (symmetric solver) and the reconstruction S = V diag(lambda) V^T
w, V = np.linalg.eigh(S_tr)
w, V = w[::-1], V[:, ::-1]                              # eigh returns ascending; we want biggest first
print("eigenvalues:", np.round(w, 3), " sum =", round(w.sum(), 3), "(= total variance = 12 tests x 1)")
print("V^T V = I ?", np.allclose(V.T @ V, np.eye(12)), "| S = V L V^T ?", np.allclose(V @ np.diag(w) @ V.T, S_tr))

ks = np.arange(1, 13)
err = [np.linalg.norm(S_tr - V[:, :k] @ np.diag(w[:k]) @ V[:, :k].T) / np.linalg.norm(S_tr) for k in ks]
fig, ax = plt.subplots(figsize=(12, 5.5))
ax.plot(ks, err, "o-", color=PALETTE[0])
ax.set_xlabel("number of eigen-directions kept"); ax.set_ylabel("relative error rebuilding the covariance")
ax.set_title("Only the first direction is cheap to keep: eleven near-equal ones follow")
plt.show()

---
## 6. Experiments & Variations

Three pre-registered experiments, each a sweep or a comparison, all with the eigenvectors learned on the 1,400 training patients.

### Experiment 1: eigenvectors of the biomarker covariance
### Experiment 2: rotate the panel, eigenvectors follow
### Experiment 3: PCA is eigendecomposition — prove it

In [ ]:
# VISUAL 3 — Experiment 1: eigenvectors of the biomarker covariance   # CENTRAL
frac = w / w.sum()
fig, (a, b) = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1, 1.3]})
a.bar(ks, w, color=[PALETTE[1] if i in (0, 11) else PALETTE[7] for i in range(12)])
a.axhline(1.0, color="k", ls="--", lw=1, label="1.0 = what 12 uncorrelated tests would each give")
a.set_xlabel("eigen-direction (largest first)"); a.set_ylabel("eigenvalue = variance along it")
a.set_title("One direction stretched to 2.1, one squashed to nearly zero: the duplicated test")
for i in (0, 11): a.text(ks[i], w[i] + 0.04, f"{w[i]:.2f}", ha="center")
a.legend(loc="upper right")
load = pd.DataFrame(V[:, [0, 1, 11]], index=[x[3:] for x in BM], columns=["1st (largest)", "2nd", "12th (near zero)"])
load.plot.bar(ax=b, color=[PALETTE[1], PALETTE[0], PALETTE[2]], width=0.8)
b.axhline(0, color="k", lw=0.8)
b.set_xlabel("test number"); b.set_ylabel("loading (weight of that test)")
b.set_title("The largest direction loads on 01 and 12; the near-zero one is 'bm_01 minus bm_12'")
plt.tight_layout(); plt.show()
print(f"1st eigenvector, loadings on 01 and 12: {V[0,0]:.3f}, {V[11,0]:.3f} (1/sqrt(2) = {1/np.sqrt(2):.3f})")
print(f"1st eigenvalue {w[0]:.3f} vs 1 + corr(01,12) = {1 + np.corrcoef(Ztr[:,0], Ztr[:,11])[0,1]:.3f}")
print(f"12th eigenvalue {w[11]:.2e}; loadings on 01 and 12: {V[0,11]:.3f}, {V[11,11]:.3f}")
print("truth says the collinear pair is", truth["collinear"], "; strong tests", truth["strong"])
print(f"share of variance in the first direction: {frac[0]:.1%}; the next eleven: {frac[1:].min():.1%} to {frac[1:].max():.1%} each")
print(f"1st eigenvector loading on strong tests 01/02: {V[0,0]:.2f}, {V[1,0]:.2f}; on the five noise-heavy tests 06-10: {np.round(V[5:10,0],2)}")

🗣️ **Plain** — The covariance has one long axis, stretched to 2.11 against the 1.0 each test would give if the twelve were unrelated, and it is made almost entirely of `bm_01` and its copy `bm_12` (loadings -0.67 each). At the other end, one axis is squashed to 0.004: it is "`bm_01` minus `bm_12`", a combination on which patients hardly differ because the two tests are the same measurement. The other ten axes are all between 0.82 and 1.12, meaning there is no further structure worth naming. This matches the generator's `truth` exactly: it said the duplicated pair was `bm_12` and `bm_01`.  
💻 **Developer** — `np.linalg.eigh(np.cov(Z_train.T))`, flip to descending, read `V[:, j]` as the loadings of direction $j$. The loadings on `bm_01` and `bm_12` in the last direction are $\mp0.707=1/\sqrt2$, which is the signature of an exact difference of two near-copies.  
🔬 **Technical** — For two standardised variables with correlation $\rho$, the eigenvalues of their $2\times2$ correlation matrix are $1\pm\rho$ with eigenvectors $(1,\pm1)/\sqrt2$. With $\rho\approx0.996$ that predicts 1.996 and 0.004. The observed smallest eigenvalue is 0.004 and the largest is 2.11, slightly above 1.996 because `bm_01` also correlates (about 0.2) with `bm_02`, the second strong test, so the first direction leans a little towards it (-0.27). The sum of the eigenvalues is 12.01, the trace of the correlation matrix. A near-zero eigenvalue means $S$ is nearly singular, which is why the normal equations of Notebook 01 became unstable.

**Takeaway for a clinician:** the decomposition found the redundant test without being told about it.

In [ ]:
# VISUAL 4 — Experiment 2: rotate the panel, eigenvectors follow
# Part A: rotate a two-test plane by an angle and watch the leading eigenvector's own angle.
pair = Ztr[:, [1, 2]]                                    # bm_02 and bm_03: two tests that are NOT duplicates
# give the pair a clear long axis so the angle is well defined
pair = pair @ np.diag([1.8, 0.6])
angles = np.arange(0, 181, 15)
lead_angle, eig_pair = [], []
for deg in angles:
    t = np.deg2rad(deg)
    Q = np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])   # rotation matrix: orthogonal, det = +1
    ww, VV = np.linalg.eigh(np.cov((pair @ Q.T).T))
    v = VV[:, -1]
    lead_angle.append(np.rad2deg(np.arctan2(v[1], v[0])) % 180)       # an eigenvector has no sign: angle mod 180
    eig_pair.append(ww[::-1])
eig_pair = np.array(eig_pair)

# Part B: a random orthogonal rotation of ALL 12 tests
rng = np.random.default_rng(SEED)
Qr, _ = np.linalg.qr(rng.normal(size=(12, 12)))         # random orthogonal matrix (a rotation or reflection of the 12-D panel)
w_rot, V_rot = np.linalg.eigh(np.cov((Ztr @ Qr).T)); w_rot, V_rot = w_rot[::-1], V_rot[:, ::-1]
err_val = np.abs(w_rot - w).max()
follow = np.abs((V_rot * (Qr.T @ V)).sum(0))             # |cos| between rotated-panel eigenvector and Q^T v
print(f"12-D rotation: max eigenvalue change {err_val:.1e}; min |cos| of eigenvectors with Q^T v over the 11 separated ones {follow[:11].min():.6f}")

# Part C: a NON-orthogonal change (rescale one test x3) is not a rotation
Zs = Ztr.copy(); Zs[:, 1] *= 3
w_sc, V_sc = np.linalg.eigh(np.cov(Zs.T)); w_sc = w_sc[::-1]; V_sc = V_sc[:, ::-1]
print(f"rescaling bm_02 x3: top eigenvalue {w[0]:.2f} -> {w_sc[0]:.2f}; top eigenvector now loads {V_sc[1,0]:.2f} on bm_02")

fig, (a, b, c) = plt.subplots(1, 3, figsize=(18, 5.2))
a.plot(angles, lead_angle, "o-", color=PALETTE[0], label="leading eigenvector's angle")
a.plot(angles, (angles + np.rad2deg(np.arctan2(*np.linalg.eigh(np.cov(pair.T))[1][::-1, -1]))) % 180, "--", color=PALETTE[1], label="start angle + rotation")
a.set_xlabel("rotation applied to the pair (degrees)"); a.set_ylabel("direction of the long axis (degrees)")
a.set_title("The long axis turns exactly as far as the data are turned"); a.legend()
b.plot(angles, eig_pair[:, 0], "o-", color=PALETTE[2], label="larger eigenvalue")
b.plot(angles, eig_pair[:, 1], "s-", color=PALETTE[3], label="smaller eigenvalue")
b.set_ylim(0, eig_pair.max() * 1.2)
b.set_xlabel("rotation applied to the pair (degrees)"); b.set_ylabel("eigenvalue = variance")
b.set_title("The eigenvalues do not move at all under rotation"); b.legend()
c.bar(np.arange(1, 13) - 0.2, w, 0.4, color=PALETTE[7], label="original panel")
c.bar(np.arange(1, 13) + 0.2, w_sc, 0.4, color=PALETTE[1], label="bm_02 rescaled x3 (not a rotation)")
c.set_xlabel("eigen-direction"); c.set_ylabel("eigenvalue")
c.set_title("Rescaling one test, unlike rotating, changes the eigenvalues"); c.legend()
plt.tight_layout(); plt.show()

🗣️ **Plain** — Imagine every patient's results are turned through some angle, as if the panel were read in a different set of units that mix tests together. The cloud's shape has not changed, so its long and short axes must turn by exactly that angle, and the lengths of the axes must stay the same. Both are what we see: the left panel shows the long axis following the rotation, and the middle panel shows two flat lines. The same holds for all twelve tests turned at once: eigenvalues changed by at most 3e-15, and the eigenvectors equal the rotated originals (|cos| ≥ 0.999999). The right panel is the contrast. Stretching a single test by three is *not* a rotation, and it changes everything: the top eigenvalue goes from 2.11 to 9.14 and the top axis becomes almost entirely `bm_02` (loading -0.99). Eigenvectors follow rotations, not rescalings.  
🔬 **Technical** — If $Z'=ZQ$ with $Q^\top Q=I$, then $S'=Q^\top SQ$, so $S'(Q^\top v)=Q^\top S v=\lambda\,Q^\top v$: eigenvalues are invariant and eigenvectors transform by $Q^\top$ (a similarity transformation). For a general invertible $D$, $S'=D^\top SD$ is a *congruence*, which preserves definiteness (Sylvester) but not eigenvalues. This is why standardising matters: choosing units is choosing $D$.

**Takeaway for a clinician:** the "main direction" is a property of the patients, not of the way the tests are written down, *provided* the change of coordinates is a rotation. Change the units and it moves.

In [ ]:
# VISUAL 5 — Experiment 3: PCA is eigendecomposition — prove it
pca = PCA(n_components=12, svd_solver="full").fit(Ztr)  # sklearn: centres, then takes an SVD of the data matrix
Zc = Ztr - Ztr.mean(0)
U, sv, Vt = np.linalg.svd(Zc, full_matrices=False)       # third route: SVD of the centred data
w_svd = sv**2 / (len(Zc) - 1)                            # squared singular values / (n-1) = eigenvalues
print("max |eigh - PCA.explained_variance_| :", np.abs(w - pca.explained_variance_).max())
print("max |eigh - SVD sv^2/(n-1)|           :", np.abs(w - w_svd).max())
cos_pca = np.abs((V * pca.components_.T).sum(0))         # |cos| between my eigenvectors and sklearn's components
scores_eig = Zte @ V                                     # project TEST patients on eigenvectors learned from TRAIN
scores_pca = pca.transform(Zte)
sign_flip = np.sign((scores_eig * scores_pca).sum(0))
print("min |cos| over 11 separated components :", cos_pca[:11].min(), "| component 12 (zero eigenvalue): ", round(cos_pca[11], 3))
print("test-score max difference after fixing signs:", np.abs(scores_eig * sign_flip - scores_pca)[:, :11].max())

fig, (a, b, c) = plt.subplots(1, 3, figsize=(18, 5.2))
a.scatter(w, pca.explained_variance_, s=70, color=PALETTE[0], label="sklearn PCA", zorder=3)
a.scatter(w, w_svd, s=25, color=PALETTE[1], label="SVD of the data", zorder=4)
a.plot([0, 2.2], [0, 2.2], "k--", lw=1, label="identity")
a.set_xlabel("eigenvalue from eigh(covariance)"); a.set_ylabel("PCA / SVD variance")
a.set_title("Three routes, one set of 12 numbers"); a.legend()
b.bar(np.arange(1, 13), cos_pca, color=PALETTE[2])
b.set_ylim(0, 1.1); b.set_xlabel("component"); b.set_ylabel("|cos| between the two sets of directions")
b.set_title("Same directions up to sign; the near-zero one is the least certain"); 
c.scatter(scores_eig[:, 0] * sign_flip[0], scores_pca[:, 0], s=8, color=PALETTE[3], alpha=0.6)
c.set_xlabel("test-patient score, eigh route"); c.set_ylabel("test-patient score, PCA route")
c.set_title("Test patients land in the same place by either route")
plt.tight_layout(); plt.show()

🗣️ **Plain** — Three different recipes, one answer. The textbook recipe (eigenvectors of the covariance), the software recipe (scikit-learn's PCA) and the numerical recipe (the SVD of the data) agree on every eigenvalue to within 3e-15 and on every direction to within sign (|cos| = 1 for the eleven separated directions). Applied to the 600 test patients, who played no part in finding the directions, the first two recipes put each patient at the same place to 2e-13 once the arbitrary signs are matched. So "principal component analysis" is nothing more than this notebook's decomposition, followed by reading the patients' coordinates along the eigenvectors. The only place the recipes could disagree is the near-zero direction, whose sign and exact orientation are the least determined, and here even that one agrees.  
🔬 **Technical** — With $Z_c=U\Sigma W^\top$, $S=W\Sigma^2W^\top/(n-1)$, so PCA's `components_` equal the rows of $W^\top$ and `explained_variance_` equals $\sigma_j^2/(n-1)$. Scores for new data are $(z-\bar z_{\text{train}})V$; the signs of the columns of $V$ are implementation-dependent, so equality is checked up to a per-column $\pm1$. scikit-learn uses the SVD route because forming $Z^\top Z$ squares the condition number, losing roughly half the significant digits for small singular values.

**Takeaway for a clinician:** when someone says "we ran PCA", they computed these eigenvectors. Nothing hidden, nothing about disease.

---
## 7. Failure Modes & Edge Cases

Three deliberate breaks:
1. **Leave the units alone.** Add age in years to the panel and take the eigenvectors of the raw covariance.
2. **Too few patients for crowded eigenvalues.** Resample 100, 300 and 1,400 training patients and ask how closely each sample eigenvector agrees with the full-data one.
3. **Treat variance as signal.** Score the test patients on each eigen-direction alone and ask which directions carry the disease, with and without the duplicate test.

In [ ]:
# VISUAL — deliberately break the method: (A) leave age in years, (B) too few patients for crowded eigenvalues, (C) variance is not disease
# A: add age (in years, SD ~ 12) to the panel and do NOT standardise
age = df["age"].to_numpy()
raw_aug = np.column_stack([X_raw, age])[idx_tr]
wA, VA = np.linalg.eigh(np.cov(raw_aug.T)); wA, VA = wA[::-1], VA[:, ::-1]
print(f"A: age SD = {age.std():.1f}; unstandardised top eigenvalue {wA[0]:.0f}, loading on age {abs(VA[12,0]):.3f}")

# B: how stable is each eigenvector when the training patients are resampled?
def stability(n, reps=60):
    rng = np.random.default_rng(SEED); out = np.zeros((reps, 12))
    for r in range(reps):
        sub = Ztr[rng.choice(len(Ztr), n, replace=True)]
        _, Vb = np.linalg.eigh(np.cov(sub.T)); Vb = Vb[:, ::-1]
        out[r] = np.abs((Vb * V).sum(0))
    return out.mean(0)
stab = {n: stability(n) for n in (100, 300, len(Ztr))}

# C: does a bigger eigenvalue mean more disease signal? held-out AUC of each single component score
auc_pc = []
for k in range(12):
    lr = LogisticRegression().fit((Ztr @ V)[:, [k]], ytr)
    p = lr.predict_proba((Zte @ V)[:, [k]])[:, 1]
    auc_pc.append(roc_auc_score(yte, p))
auc_pc = np.array(auc_pc)
# same question with the duplicate (bm_12) removed from the panel
keep = list(range(11)); w11, V11 = np.linalg.eigh(np.cov(Ztr[:, keep].T)); V11 = V11[:, ::-1]
auc_pc11 = np.array([roc_auc_score(yte, LogisticRegression().fit((Ztr[:, keep] @ V11)[:, [k]], ytr).predict_proba((Zte[:, keep] @ V11)[:, [k]])[:, 1]) for k in range(11)])
print("C: held-out AUC per component, 12 tests:", np.round(auc_pc, 3))
print("C: held-out AUC per component, duplicate removed:", np.round(auc_pc11, 3), "| top eigenvalue now", round(w11[-1], 3))

fig, (a, b, c) = plt.subplots(1, 3, figsize=(18, 5.2))
a.bar(np.arange(1, 14), np.abs(VA[:, 0]), color=[PALETTE[1] if i == 12 else PALETTE[7] for i in range(13)])
a.set_xticks(np.arange(1, 14)); a.set_xticklabels([str(i) for i in range(1, 13)] + ["age"])
a.set_xlabel("variable"); a.set_ylabel("|loading| on the top eigenvector")
a.set_title("Age in years swamps the panel: the top direction is just age")
for n, col_, mk in zip(stab, [PALETTE[1], PALETTE[4], PALETTE[0]], ["o", "s", "^"]):
    b.plot(np.arange(1, 13), stab[n], marker=mk, color=col_, label=f"{n} patients")
b.set_xlabel("eigen-direction (largest first)"); b.set_ylabel("mean |cos| with the full-data direction")
b.set_title("Only the top direction is stable; crowded ones wander"); b.legend(); b.set_ylim(0, 1.05)
c.bar(np.arange(1, 13) - 0.2, auc_pc - 0.5, 0.4, color=PALETTE[0], label="all 12 tests")
c.bar(np.arange(1, 12) + 0.2, auc_pc11 - 0.5, 0.4, color=PALETTE[4], label="duplicate bm_12 removed")
c.set_xlabel("eigen-direction (largest first)"); c.set_ylabel("held-out AUC minus 0.5")
c.set_title("The biggest direction is the disease signal, with or without the duplicate"); c.legend()
plt.tight_layout(); plt.show()
print("B: stability of direction 1 at 100/300/full:", [round(stab[n][0], 3) for n in stab], "; direction 12:", [round(stab[n][11], 3) for n in stab])
print("B: mean |cos| of components 2-11 at 100 / 300 / full:", [round(stab[n][1:11].mean(), 3) for n in stab])

🗣️ **Plain** — Break 1: with age in years in the mix, the "main pattern" is simply age, because years have a bigger spread than the SD-unit tests, and the method faithfully reports the units. Break 2: with 100 patients the main direction is only roughly reproducible (mean |cos| 0.90) and by 1,400 it is near-perfect (0.99), but the ten crowded directions never settle (0.30 to 0.46): it is like asking which way a round ball's "long axis" points. Break 3: the biggest direction scored a held-out AUC of 0.88 and the others were close to chance (0.45 to 0.61). That is a fact about this generator, not about eigenvectors.  
💻 **Developer** — Guard rails: standardise inside the training fold, check the gap $\lambda_j-\lambda_{j+1}$ before interpreting direction $j$, bootstrap the eigenvectors (as above) before naming them, and evaluate any use of the scores on held-out patients.  
🔬 **Technical** — Davis–Kahan: the sine of the angle between a sample and a population eigenvector is bounded by roughly $\|S-\Sigma\|/\text{gap}$. With gaps of 0.01–0.05 between adjacent eigenvalues and sampling error of order $\lambda\sqrt{2/n}\approx0.04$ at $n=1400$, the middle eigenvectors are not identified; only $\lambda_1$ (gap ≈ 1.0) and $\lambda_{12}$ (gap ≈ 0.8) are.

### (exploratory) Top-k eigen-directions against the full panel

Not pre-registered: findings from this figure are hypothesis-generating only and feed no conclusion in sections 8–9.

In [ ]:
# Results figure: does keeping the top-k eigen-directions keep the disease signal? (train only to fit, test to score)
rows = []
for k in range(1, 13):
    lr = LogisticRegression(max_iter=1000).fit(Ztr @ V[:, :k], ytr)
    rows.append(roc_auc_score(yte, lr.predict_proba(Zte @ V[:, :k])[:, 1]))
lr_all = LogisticRegression(max_iter=1000).fit(Ztr, ytr)
auc_all = roc_auc_score(yte, lr_all.predict_proba(Zte)[:, 1])
strong_idx = [BM.index(t) for t in truth["strong"]]
auc_strong = roc_auc_score(yte, LogisticRegression().fit(Ztr[:, strong_idx], ytr).predict_proba(Zte[:, strong_idx])[:, 1])
# bootstrap range for the all-tests AUC
rng = np.random.default_rng(SEED); pp = lr_all.predict_proba(Zte)[:, 1]
bs = [roc_auc_score(yte[i], pp[i]) for i in (rng.choice(len(yte), len(yte)) for _ in range(500)) if yte[i].min() != yte[i].max()]
lo, hi = np.percentile(bs, [2.5, 97.5])
print(f"all 12 tests: {auc_all:.3f} (bootstrap {lo:.3f}-{hi:.3f}); two strong tests: {auc_strong:.3f}")
print("top-k eigen-directions:", np.round(rows, 3))

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.plot(range(1, 13), rows, "o-", color=PALETTE[0], label="logistic regression on top-k eigen-direction scores")
ax.axhline(auc_all, color=PALETTE[2], ls="--", label=f"all 12 tests directly ({auc_all:.2f})")
ax.axhline(auc_strong, color=PALETTE[1], ls=":", label=f"only the two strong tests ({auc_strong:.2f})")
ax.axhspan(lo, hi, color=PALETTE[2], alpha=0.15, label="bootstrap range, all 12 tests")
ax.set_xlabel("eigen-directions kept (largest variance first)"); ax.set_ylabel("held-out AUC")
ax.set_title("The first direction alone matches the whole panel; the rest add nothing")
ax.legend(loc="lower right"); plt.show()

---
## 8. Results, Findings & Implications

### 📊 Results

| Experiment | Question | Result |
|---|---|---|
| 1. Eigenvectors of the covariance | What does the 12 x 12 panel decompose into? | $\lambda_1$ = 2.11 (loads -0.67 on `bm_01` and `bm_12`), $\lambda_{12}$ = 0.004 (±0.707 on the same pair), the middle ten between 0.82 and 1.12 |
| 2. Rotate the panel | Do eigenvectors follow? | Eigenvalues unchanged (≤ 3e-15); eigenvectors rotate with the data; rescaling one test x3 moves $\lambda_1$ from 2.11 to 9.14 |
| 3. PCA = eigendecomposition | Same thing? | eigh vs PCA vs SVD agree to 3e-15; test-patient scores agree to 2e-13 |
| Failure 1 | Age in years | top eigenvector is 100% age (loading 1.000) |
| Failure 2 | Resampling | direction 1: mean \|cos\| 0.90 / 0.97 / 0.99 at 100 / 300 / 1,400 patients; directions 2–11: 0.30 / 0.33 / 0.46 |
| Failure 3 | Variance vs disease | held-out AUC of direction 1 alone 0.88 (0.87 with the duplicate removed); no other direction above 0.61 |

The figure just above (end of section 7) is an *exploratory* extra, not one of the three pre-registered experiments; it is not used for any finding below.

### 🔍 Findings

1. **The decomposition recovered the generator's duplicated test without being told.** The near-zero eigenvalue (0.004) and its ±0.707 loadings on `bm_01` and `bm_12` match `truth["collinear"]`.
2. **Rotation leaves the eigenvalues alone and turns the eigenvectors with the data; rescaling changes both.** Rotation left eigenvalues identical to 3e-15. Rescaling one test (a change of units) changed the top eigenvalue from 2.11 to 9.14 and the top eigenvector, so eigenvalues depend on units too.
3. **PCA adds nothing beyond this decomposition.** Three routes gave the same eigenvalues to 3e-15.
4. **Unstandardised data return the units, not the biology.** With age in years in the panel the top eigenvector is entirely age (eigenvalue 192 against 2.1).
5. **Only the top direction is stable at realistic sample sizes.** Directions 2–11, whose eigenvalues sit within 0.82–1.12, have mean \|cos\| of just 0.46 with their full-data versions even when resampling all 1,400 patients. Their order and orientation are not reproducible. (The near-zero direction is perfectly stable, at 1.0.)
6. **Surprise: the biggest direction predicted well, and the decomposition did not look at the diagnosis.** Alone it scores a held-out AUC of 0.88 (0.87 with the duplicate removed); the second direction scores 0.58, near chance. Almost all of $\lambda_1$ (2.11) comes from the duplicated pair (1 + 0.996 = 1.996 would be the pair alone), and that pair includes a strong test, `bm_01`. We did not test why the first direction predicts well beyond that, and under the `predict` goal this is an observation about this panel, not a mechanism. Whether the largest direction predicts anything depends on the panel; here it did.

### 💡 Implications

- **Use** the eigen-decomposition of the standardised covariance to look for redundant tests (near-zero eigenvalues), to see how many genuinely separate directions a panel has, and as the first step of PCA.
- **Avoid** reading the lower eigenvectors individually when eigenvalues are crowded and $n$ is modest: they are an arbitrary basis for a flat subspace.
- **Do not** let variance stand in for relevance. Whether the large direction carries disease depends on the disease. Keep the supervised check (held-out AUC) separate.
- **Standardise first, on the training patients**, and apply the training means, SDs and eigenvectors to new patients.

### ⚠️ Considerations

- **Assumptions**: variance is the thing of interest; relationships are summarised by covariance (linear); the training cloud resembles future patients.
- **Computational cost**: forming the covariance $O(nd^2)$, `eigh` $O(d^3)$. At 12 tests it is instant; for thousands of tests use truncated SVD or power-style methods.
- **Scaling**: the decomposition depends on units. Standardise (or use the correlation matrix) unless the tests share meaningful units.
- **Common mistakes**: comparing eigenvectors without fixing signs; interpreting directions inside a cluster of near-equal eigenvalues; fitting the eigenvectors on all patients including the test set; assuming the top direction is the disease direction.
- **Ethics / safety**: if a subgroup (for example by age or sex) is under-represented in the training patients, its structure may not appear in the main directions, and a screening tool built on them can quietly fit that subgroup worse. Check subgroups before use.

---
## 9. Clinical Reading

_Does this person have the condition? How sure are we?_

**What this means in the clinic.** The eigen-decomposition does not diagnose anyone. It tells the service how many separate pieces of information its twelve-test panel really contains. On these synthetic patients that is eleven, not twelve: one test is a duplicate of another (its direction has variance 0.004). Scoring 600 unseen patients on the largest training direction alone gave a held-out AUC of 0.88, but that is one split without a spread, so treat it as indicative. "How sure are we" is therefore still the sure-ness of the downstream model; this step contributes a simpler description of the inputs, not a certainty.

**What a clinician would do differently having seen this:**
- Ask whether the panel pays for a test twice. A direction with near-zero variance means one test could be dropped with almost no loss of information, saving cost and a venepuncture.
- Ask what units a "main pattern" was computed in before trusting it. With age in years left in, the main pattern was just age.
- Treat the leading pattern as a candidate summary score to validate, not as a result.

**What they must not conclude:**
- That the largest direction *is* the disease, or that it predicts well for a reason we understand. Here it predicted well (held-out AUC 0.88) and we know only that the duplicated pair includes a strong test. In a real service the largest direction is often age, sex, a batch or a machine effect. The decomposition never sees the diagnosis.
- That the lower directions mean anything individually. Ten of them have eigenvalues between 0.82 and 1.12 and are not reproducible when patients are resampled.
- That a test with a large loading causes the condition or is the most important test. Under this notebook's `predict` goal, loadings describe variance, not mechanism.
- That these numbers transfer to a real service. These are **retrospective results on synthetic patients** from a generator built to contain a duplicated test. A real panel needs a prospective evaluation on its own patients, with sensitivity and specificity reported at a threshold chosen in advance.

---
## 10. Connections

- **Builds on**: 00.01.01 *Vectors, Matrices & Operations*: the duplicated test and its singular-value signature, here seen as an eigenvalue.
- **Leads to**:
  - 00.01.03 *SVD & Low-Rank Approximation*: the same decomposition for rectangular data, and why `sklearn` uses it;
  - 00.06.01 *Stability, Precision & Solvers*: the nearly singular covariance and the cost of squaring the condition number;
  - principal component analysis in the dimensionality-reduction module, built directly on Experiment 3;
  - 04.02.02 *Multicollinearity & VIF*: the near-zero eigenvalue under an `explain` goal.
- **Alternatives**:
  - SVD of the data matrix, which avoids forming the covariance;
  - factor analysis, which models the noise explicitly;
  - supervised directions (linear discriminant analysis, partial least squares), which use the diagnosis;
  - robust covariance estimators when outliers dominate.

---
## 11. Key Takeaways

1. An eigenvector of the covariance is a direction in which the patient cloud is only stretched, and its eigenvalue is the variance along it; $S=V\Lambda V^\top$ rebuilds the covariance exactly.
2. A near-zero eigenvalue is a redundant test, found here without being told (0.004, loadings ±0.707 on `bm_01` and `bm_12`).
3. Rotating the panel rotates the eigenvectors and leaves the eigenvalues alone; rescaling a test changes both, so standardise on training patients first.
4. PCA, `eigh` of the covariance and the SVD of the data are one computation (agreement to 3e-15).
5. Crowded eigenvalues give unreproducible directions, and large variance is not disease signal: the decomposition never sees the diagnosis.

---
## References

- Deisenroth, Faisal, Ong (2020). *Mathematics for Machine Learning*. `library:deisenroth2020`